In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from ocr_vs_vlm.results_final.shared.colors import get_model_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import bootstrap_ci, wilcoxon_test, cohens_d, compute_significance_matrix
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template, create_metric_boxplot, create_radar_chart
from ocr_vs_vlm.results_final.shared.data_loader import load_dataset_data, extract_models_from_columns, extract_metric_scores, PHASE_TO_APPROACH

setup_plotly_template()
print("✓ Setup complete")

## 1. Load All DocVQA Experiments

In [ ]:
DATASET = 'DocVQA_mini'

try:
    df = load_dataset_data(DATASET)
    print(f"✓ Loaded {len(df)} total samples")
    print(f"  Phases: {df['phase'].unique().tolist()}")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

In [ ]:
# Summary by phase
models = extract_models_from_columns(df)
print(f"Models available: {models}")

phase_summary = []
for phase in df['phase'].unique():
    phase_df = df[df['phase'] == phase]
    for model in models:
        col = f'anls_score_{model}'
        if col in phase_df.columns:
            scores = phase_df[col].dropna()
            if len(scores) > 0:
                phase_summary.append({
                    'Phase': phase,
                    'Approach': PHASE_TO_APPROACH.get(phase, 'unknown'),
                    'Model': model,
                    'ANLS': scores.mean(),
                    'N': len(scores)
                })

summary_df = pd.DataFrame(phase_summary)
if len(summary_df) > 0:
    display(summary_df.pivot_table(index=['Phase', 'Approach'], columns='Model', values='ANLS').round(4))

## 2. Approach Comparison

In [ ]:
# Aggregate by approach
approach_stats = []
for approach in df['approach'].unique():
    app_df = df[df['approach'] == approach]
    for model in models:
        col = f'anls_score_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                approach_stats.append({
                    'Approach': approach,
                    'Model': model,
                    'ANLS': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi
                })

app_df = pd.DataFrame(approach_stats)

# Visualization
if len(app_df) > 0:
    fig = go.Figure()
    for approach in app_df['Approach'].unique():
        data = app_df[app_df['Approach'] == approach]
        color = APPROACH_COLORS.get(approach, '#6B7280')
        fig.add_trace(go.Bar(
            x=data['Model'],
            y=data['ANLS'],
            name=approach,
            marker_color=color,
            error_y=dict(type='data', array=data['ANLS'] - data['CI_Lower'])
        ))
    
    fig.update_layout(
        title='DocVQA: ANLS by Approach',
        yaxis_title='ANLS Score',
        barmode='group',
        height=500
    )
    fig.show()

## 3. Statistical Significance

In [ ]:
# Test: OCR Pipeline vs Direct VQA
ocr_scores = df[df['approach'] == 'ocr_pipeline']
direct_scores = df[df['approach'] == 'direct_vqa']

print("📊 Approach Comparison: OCR Pipeline vs Direct VQA")
print("-" * 60)

for model in models:
    col = f'anls_score_{model}'
    if col in ocr_scores.columns and col in direct_scores.columns:
        ocr = ocr_scores[col].dropna().values
        direct = direct_scores[col].dropna().values
        
        if len(ocr) > 0 and len(direct) > 0:
            # Match sample sizes by taking minimum
            n = min(len(ocr), len(direct))
            stat, p = wilcoxon_test(ocr[:n], direct[:n])
            d = cohens_d(ocr, direct)
            
            ocr_mean = np.mean(ocr)
            direct_mean = np.mean(direct)
            winner = "OCR" if ocr_mean > direct_mean else "Direct"
            
            print(f"{model}:")
            print(f"   OCR: {ocr_mean:.4f} | Direct: {direct_mean:.4f} | Winner: {winner}")
            print(f"   p={p:.4f}, Cohen's d={d:.3f}")

## 4. Interactive Explorer

In [ ]:
# Interactive widget for filtering
phase_selector = widgets.SelectMultiple(
    options=df['phase'].unique().tolist() if len(df) > 0 else [],
    value=df['phase'].unique().tolist()[:3] if len(df) > 0 else [],
    description='Phases:',
    layout=widgets.Layout(width='300px', height='150px')
)

model_selector = widgets.SelectMultiple(
    options=models,
    value=models[:2] if len(models) >= 2 else models,
    description='Models:',
    layout=widgets.Layout(width='300px', height='100px')
)

def update_plot(phases, models_selected):
    filtered = df[df['phase'].isin(phases)]
    if len(filtered) == 0:
        print("No data for selected filters")
        return
    
    data = {}
    for model in models_selected:
        col = f'anls_score_{model}'
        if col in filtered.columns:
            data[model] = filtered[col].dropna().values
    
    if data:
        fig = create_metric_boxplot(data, 'ANLS', f'DocVQA: {len(phases)} phases')
        fig.show()

widgets.interactive(update_plot, phases=phase_selector, models_selected=model_selector)

## 5. Key Findings

In [ ]:
print("=" * 70)
print("DOCVQA MINI - KEY FINDINGS")
print("=" * 70)

if len(app_df) > 0:
    # Best by approach
    print("\n📊 Best ANLS by Approach:")
    for approach in app_df['Approach'].unique():
        best = app_df[app_df['Approach'] == approach].nlargest(1, 'ANLS').iloc[0]
        print(f"   {approach}: {best['Model']} → {best['ANLS']:.4f}")
    
    # Overall best
    overall = app_df.nlargest(1, 'ANLS').iloc[0]
    print(f"\n🏆 Best Overall: {overall['Approach']} + {overall['Model']} → ANLS={overall['ANLS']:.4f}")

print("\n" + "=" * 70)